# Week 9: Transformers & LLM Architectures
## Notebook 10: SwiGLU: Swish-Gated Linear Units in Modern LLMs

### 1. Conceptual Intuition & Theoretical Background
Early Transformers (e.g. original Vaswani, GPT-2) used standard two-layer Feed-Forward Networks (FFN) with ReLU or GELU activations:
$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

**Gated Linear Units (GLU, Dauphin et al., 2016):**
GLUs use a **bilinear gating mechanism**: the input is projected along two parallel linear pathways, where one path gates the other via an element-wise product:
$$\text{GLU}(x, W, V) = \sigma(x W) \otimes (x V)$$

**SwiGLU (Shazeer, 2020):**
Replaces the sigmoid gate with the **Swish/SiLU** activation ($\text{Swish}(z) = z \cdot \sigma(z)$):
$$\text{SwiGLU}(x) = \left( x W_1 \cdot \sigma(x W_1) \right) \otimes (x W_{\text{gate}})$$
$$\text{FFN}_{\text{SwiGLU}}(x) = \left( \text{Swish}(x W_1) \otimes (x W_{\text{gate}}) \right) W_2$$

Adopted by **LLaMA-1/2/3, Mistral, Gemma, and PaLM** because it yields lower perplexity per training step.

---

### 2. Scratch Python Implementation: SwiGLU Feedforward Layer

In [ ]:
import numpy as np

class SwiGLUFFNScratch:
    def __init__(self, d_model=64, d_ff=170):
        """
        To maintain equivalent parameter count to standard FFN with 4 * d_model,
        SwiGLU sets d_ff ~ (8/3) * d_model.
        """
        self.d_model = d_model
        self.d_ff = d_ff
        
        # Xavier initialization
        std = np.sqrt(2.0 / (d_model + d_ff))
        self.W1 = np.random.normal(0, std, (d_model, d_ff))       # Up projection
        self.W_gate = np.random.normal(0, std, (d_model, d_ff))   # Gate projection
        self.W2 = np.random.normal(0, std, (d_ff, d_model))       # Down projection
        
    def _swish(self, x):
        return x / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))
        
    def forward(self, x):
        """x: shape (batch_size, seq_len, d_model)"""
        # Parallel projections
        gate = np.dot(x, self.W_gate)
        up = np.dot(x, self.W1)
        
        # SwiGLU element-wise gating: Swish(gate) * up
        swiglu_out = self._swish(gate) * up
        
        # Down-projection back to d_model
        out = np.dot(swiglu_out, self.W2)
        return out

# Test SwiGLU FFN on synthetic hidden states
batch, seq_len, d_model = 2, 8, 64
d_ff = int((8 / 3) * d_model) # ~170

ffn = SwiGLUFFNScratch(d_model=d_model, d_ff=d_ff)
h_in = np.random.randn(batch, seq_len, d_model)

h_out = ffn.forward(h_in)

print("--- SwiGLU Gated Feed-Forward Architecture ---")
print(f"Input Embedding Dimension:      {d_model}")
print(f"Intermediate Dimension (8/3 d): {d_ff}")
print(f"Input Hidden States:            {h_in.shape}")
print(f"Output Hidden States:           {h_out.shape}")
print(f"Total Weights (3 matrices):     {3 * (d_model * d_ff)} params")

### 3. Step-by-Step Code Walkthrough
1. **Bilinear Pathways:** Computes $x W_1$ and $x W_{\text{gate}}$ concurrently.
2. **Dynamic Gating:** The Swish activation controls how much information flows through the linear path.
3. **Dimension Adjustment:** Sets intermediate dimension to $\frac{8}{3} d_{\text{model}}$ so parameter count matches standard FFNs.

---

### 4. Industry Applications & Trade-offs
- **Modern LLM Architectures:** Standard in LLaMA-3, Mistral-7B, and PaLM 2.
- **Trade-off:** Requires 3 weight matrices instead of 2, increasing memory bandwidth requirements during weight loading.

---

### 5. Top Interview Questions & Answers
**Q1: Why does SwiGLU set its intermediate hidden dimension to $\frac{8}{3} d_{\text{model}}$ instead of $4 d_{\text{model}}$?**
> **Answer:** Standard FFNs have 2 matrices ($2 \times 4 d^2 = 8 d^2$). SwiGLU uses 3 matrices ($W_1, W_{\text{gate}}, W_2$). Setting intermediate dimension to $\frac{8}{3} d_{\text{model}}$ yields $3 \times \frac{8}{3} d^2 = 8 d^2$, keeping parameter count and compute flops identical to a standard FFN.

**Q2: What is the architectural intuition behind gated activations?**
> **Answer:** Gating allows the network to dynamically scale or suppress individual activation channels based on input context, acting like a data-dependent routing mechanism.